In [22]:
##### Assigns country intensities to pixels and sub-regions
# pixel and subnational (vector) scale
# variables: 
    # capital intensity (USD / USD)
    # labor intensity (jobs / million USD)

import os
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from pathlib import Path
import rasterstats
from rasterio.transform import from_bounds
from rasterio.features import rasterize
from scipy.ndimage import distance_transform_edt


In [23]:
##### Load data

# Get the current working directory
cd = Path.cwd().parent.parent 

# sub-national geographies
total_geo = gpd.read_file(f"{cd}/Data/Clean/Geographies/subnational_total.shp")
countries = gpd.read_file("/Users/carinamanitius/Documents/Data/Admin_Boundaries/gadm_410-levels.gpkg", layer="ADM_0")

# reference raster
ref_path = f"{cd}/Data/Clean/Production/total_production_USD_2020.tif"

# country intensities
intensities = pd.read_csv(f"{cd}/Data/Clean/Intensities/country_intensities.csv")
country_codes = pd.read_csv(f"{cd}/Data/Correspondence_tables/country_names.csv", encoding="cp1252")

# save paths
pixels_capital = f"{cd}/Data/Clean/Predictors/Rasters/country_capital_intensity.tif"
pixels_labor = f"{cd}/Data/Clean/Predictors/Rasters/country_labor_intensity.tif"

vectors = f"{cd}/Data/Clean/Predictors/Vectors/country_intensities.csv"

In [24]:
#### Run resampling for pixel scale 

### PREP 

# add geo to country intensities
intensities_geo = intensities.merge(country_codes, on='ISO3', how='left')
intensities_geo = intensities_geo.merge(countries, left_on='SHP_code', right_on='GID_0', how='left')

intensities_geo = intensities_geo[['ISO3', 'labor_intensity_jobs_per_million_USD', 'capital_intensity_USD_per_USD', 'geometry']]

intensities_geo = gpd.GeoDataFrame(intensities_geo, geometry="geometry", crs=countries.crs)
intensities_geo = intensities_geo[intensities_geo.geometry.notna()]

# get target grid from reference raster
with rasterio.open(ref_path) as ref:
    dst_crs       = ref.crs
    dst_transform = ref.transform
    dst_shape     = ref.shape
    dst_meta      = ref.meta.copy()

intensities_geo = intensities_geo.to_crs(dst_crs)

### RASTERIZE LABOR
# border pixels get value of country that centroid falls within

shapes = (
    (geom, value)
    for geom, value in zip(intensities_geo.geometry, intensities_geo["labor_intensity_jobs_per_million_USD"])
)

rasterized = rasterize(
    shapes=shapes,
    out_shape=dst_shape,
    transform=dst_transform,
    fill=np.nan,          
    dtype=np.float32,
    all_touched=False,    
)

# save
out_meta = dst_meta.copy()
out_meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")

out_arr = rasterized.copy()
out_arr[np.isnan(out_arr)] = -9999

with rasterio.open(pixels_labor, "w", **out_meta) as dst:
    dst.write(out_arr, 1)


### RASTERIZE CAPITAL
# border pixels get value of country that centroid falls within

shapes = (
    (geom, value)
    for geom, value in zip(intensities_geo.geometry, intensities_geo["capital_intensity_USD_per_USD"])
)

rasterized = rasterize(
    shapes=shapes,
    out_shape=dst_shape,
    transform=dst_transform,
    fill=np.nan,          
    dtype=np.float32,
    all_touched=False,    
)

# save
out_meta = dst_meta.copy()
out_meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")

out_arr = rasterized.copy()
out_arr[np.isnan(out_arr)] = -9999

with rasterio.open(pixels_capital, "w", **out_meta) as dst:
    dst.write(out_arr, 1)

In [25]:
#### Run resampling for vector scale 

### Assign calues countries to regions 

# merge 
total_geo = total_geo.merge(intensities, on='ISO3', how='left')

# filter
total_geo = total_geo[['PROJ_ID', 'labor_intensity_jobs_per_million_USD', 'capital_intensity_USD_per_USD']]

# rename
total_geo = total_geo.rename(columns={'labor_intensity_jobs_per_million_USD': 'country_labor_intensity_jobs_per_million_USD', 'capital_intensity_USD_per_USD': 'country_capital_intensity_USD_per_USD'})

# save 
total_geo.to_csv(vectors, index=False)